# Reinforcement Learning Assignment

## Taxi-v3: Q-Learning and Deep Q-Networks (Student Version)

This notebook is the **student version** of the reinforcement learning assignment.

You will:

1. Understand the **Taxi-v3** environment.
2. Implement **Q-Learning** with a tabular Q-table.
3. Implement a **Deep Q-Network (DQN)** using a simple MLP.
4. Compare learning behavior and discuss the results.

Fill in all the cells marked with **`TODO`**.


## 1. The Taxi-v3 Environment

The **Taxi-v3** environment is a 5×5 grid with 4 special locations:

- **R (Red)**
- **G (Green)**
- **Y (Yellow)**
- **B (Blue)**

The agent controls a taxi that must:

1. Navigate to the passenger location.
2. **Pick up** the passenger.
3. Navigate to the destination.
4. **Drop off** the passenger.

### State Space

Each state encodes:

- Taxi row (0–4)
- Taxi column (0–4)
- Passenger location (R, G, Y, B, or inside the taxi)
- Destination (R, G, Y, B)

In total there are **500** discrete states. 

#### Example: What does a state like `321` mean?


Taxi at (row=3, col=1) Passenger waiting at location R Destination is G

state = (((taxi_row * 5) + taxi_col) * 5 + passenger_location) * 4 + destination

With 321, we reverse the encoding:

1. `destination = 321 % 4 = 1`
2. `remaining = 321 // 4 = 80`
3. `passenger_location = 80 % 5 = 0`
4. `remaining = 80 // 5 = 16`
5. `taxi_col = 16 % 5 = 1`
6. `taxi_row = 16 // 5 = 3`

So the meaning of state **321** is:

- Taxi row → **3**
- Taxi column → **1**
- Passenger location → **0** (i.e., **Red**)
- Destination → **1** (i.e., **Green**)


You do *not* decode this manually — Gym handles it internally.  


### Actions

The action space has 6 discrete actions:

- `0` – move south  
- `1` – move north  
- `2` – move east  
- `3` – move west  
- `4` – pick up passenger  
- `5` – drop off passenger  

### Rewards

- Each step: **−1**
- Successful drop-off: **+20**
- Illegal pick-up or drop-off: **−10**

The goal is to **maximize total reward per episode** (solve the task efficiently, without illegal moves).


### 1.1 Imports and Environment Setup

Run the following cell to import the required libraries and create the Taxi-v3 environment.


In [ ]:
!pip install "gymnasium[toy_text]" torch matplotlib numpy

In [ ]:
import gymnasium as gym
import numpy as np
import random
from IPython.display import clear_output
import matplotlib.pyplot as plt
from time import sleep

env = gym.make("Taxi-v3", render_mode="rgb_array")
print("State space:", env.observation_space)
print("Action space:", env.action_space)

### Helper for graphical rendering

The following function will display RGB frames returned by `env.render()` using `matplotlib`.


In [ ]:
def show_frame(frame):
    plt.imshow(frame)
    plt.axis("off")
    plt.show()

## 2. Tabular Q-Learning on Taxi-v3

We will first implement **Q-Learning** using a Q-table of size:

$ Q \in \mathbb{R}^{500 \times 6} $

Each entry $ Q[s, a] $ stores the expected return of taking action $a$ in state $ s $ and following the current policy thereafter.

### 2.1 Initialize Q-Table and Hyperparameters

**TODO:**

- Initialize the Q-table with zeros using the environment dimensions.
- Set the Q-Learning hyperparameters:
  - Learning rate `alpha`
  - Discount factor `gamma`
  - Exploration parameters: `epsilon`, `epsilon_min`, `epsilon_decay`
  - Number of training episodes


In [ ]:
# TODO: Initialize Q-table
# Hint: use env.observation_space.n and env.action_space.n

Q = None  # replace with a numpy array of shape (env.observation_space.n, env.action_space.n) filled with zeros

alpha = 0.1      # learning rate
gamma = 0.99     # discount factor
epsilon = 1.0    # initial exploration rate
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 5000

Q, alpha, gamma, epsilon

### 2.2 Q-Learning Training Loop

We will now implement the **Q-Learning update rule**:

$
Q[s, a] \leftarrow Q[s, a] + \alpha \left(r + \gamma \max_{a'} Q[s', a'] - Q[s, a]\right)
$

At each step, we will:

1. Select an action using **ε-greedy**:
   - With probability ε: choose a random action (exploration).
   - With probability 1 − ε: choose the best action according to the current Q-table (exploitation).
2. Observe the reward and next state.
3. Update the Q-table using the equation above.
4. Decrease ε over time to reduce exploration.

We also print some feedback during training and, from a configurable episode onward, **occasionally visualize the current greedy policy**.

**TODO:**

- Implement ε-greedy action selection.
- Implement the Q-Learning update rule.
- Keep the printing and visualization code to see training feedback.


In [ ]:
rewards_per_episode = []
print_every = 500  # how often to print training feedback

# configuration for policy visualization during training
viz_start_ep = 3000   # start visualizing after this episode index (1-based)
viz_every = 500       # visualize every N episodes once viz_start_ep is reached
viz_max_steps = 30    # max steps for each visualization rollout

for ep in range(episodes):
    state, info = env.reset()
    done = False
    total_r = 0

    while not done:
        # TODO: epsilon-greedy action selection
        # if random.random() < epsilon: take random action
        # else: take action with highest Q[state, :]

        action = None  # TODO: replace with epsilon-greedy policy

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # TODO: Q-learning update rule
        # target = reward + gamma * max_a' Q[next_state, a']
        # Q[state, action] = Q[state, action] + alpha * (target - Q[state, action])

        # TODO: update Q-table here

        state = next_state
        total_r += reward

    # decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_r)

    if (ep + 1) % print_every == 0:
        recent_mean = np.mean(rewards_per_episode[-print_every:])
        print(f"Episode {ep+1}/{episodes} - epsilon={epsilon:.3f} - recent mean reward={recent_mean:.2f}")

    # occasional visualization of greedy policy during training
    if (ep + 1) >= viz_start_ep and (ep + 1) % viz_every == 0:
        print(f"\n[Q-Learning] Visualizing greedy policy at episode {ep+1}")
        v_state, v_info = env.reset()
        for t in range(viz_max_steps):
            clear_output(wait=True)
            frame = env.render()
            show_frame(frame)
            # greedy w.r.t current Q-table
            # (if Q is still all zeros, behavior will be arbitrary)
            if Q is None:
                break
            v_action = np.argmax(Q[v_state, :])
            v_state, v_reward, v_term, v_trunc, v_info = env.step(v_action)
            if v_term or v_trunc:
                break
            sleep(0.15)

plt.plot(rewards_per_episode)
plt.title("Q-Learning: Total reward per episode")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.show()

### 2.3 Visualizing the Learned Policy

Now, we will use the trained Q-table to run a single episode and **visualize** the final behavior of the agent.

**TODO:**

- In each step, choose the **best action** according to the Q-table (no exploration).
- Observe how the taxi navigates, picks up the passenger, and drops them off.


In [ ]:
state, info = env.reset()
done = False

for t in range(50):
    clear_output(wait=True)
    frame = env.render()
    show_frame(frame)

    # TODO: choose best action from Q-table (argmax over Q[state, :])
    action = None  # replace with greedy action from Q

    state, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    sleep(0.15)

## 3. Deep Q-Network (DQN)

Now we replace the Q-table with a **neural network** that approximates the Q-function:

$ Q(s, a; \theta) $

We will use a simple **Multi-Layer Perceptron (MLP)** that:

- Takes as input a **one-hot encoding** of the state (size 500).
- Outputs a vector of Q-values for each of the 6 actions.

We will train the DQN using **experience replay** and the same ε-greedy policy.


### 3.1 DQN Model Definition

Run the following cell to import PyTorch. Then, complete the `DQN` class:

**TODO:**

- Implement an MLP with:
  - Input dimension = 500 (one-hot state)
  - One hidden layer (e.g., 128 units, ReLU)
  - Output dimension = 6 (one value per action)
- Implement the `forward` method.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# TODO: Implement MLP for DQN
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define your network here
        # Example: Linear(500, 128) -> ReLU -> Linear(128, 6)
        pass

    def forward(self, x):
        # TODO: forward pass
        # Remember: x will be a 1D tensor of size 500 (state one-hot)
        pass

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

model

### 3.2 One-Hot Encoding for States

We will represent each discrete state `s` (0 to 499) as a 1D one-hot vector of size 500.

Run the following helper function.


In [ ]:
def one_hot_state(s):
    v = np.zeros(env.observation_space.n)
    v[s] = 1
    return torch.tensor(v, dtype=torch.float32)

### 3.3 DQN Training with Experience Replay

We now implement the DQN training loop.

We will:

1. Use an **ε-greedy policy** (similar to Q-Learning).
2. Store transitions in a **replay buffer** of tuples:  
   $ (s, a, r, s', done) $
3. Sample random batches from the buffer and update the network weights using:

$
y = \begin{cases}
r & \text{if } done \\
r + \gamma \max_{a'} Q(s', a'; \theta) & \text{otherwise}
\end{cases}
$

and minimize the MSE loss between:  
$ Q(s, a; \theta) $ and $ y $.

We also include print statements during training and occasional **visualization of the current greedy DQN policy** from a configurable episode onward.

**TODO:**

- Implement ε-greedy action selection using the neural network.
- Implement sampling from the replay buffer.
- Implement the target computation and backpropagation step.


In [ ]:
replay_buffer = []
buffer_size = 50000
batch_size = 64

episodes = 2000
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

reward_log = []
print_every = 10

# configuration for policy visualization during DQN training
dqn_viz_start_ep = 0   # start visualizing after this episode
dqn_viz_every = 200       # visualize every N episodes once started
dqn_viz_max_steps = 30    # max steps per visualization rollout

for ep in range(episodes):
    state, info = env.reset()
    done = False
    ep_reward = 0

    while not done:

        # TODO: epsilon-greedy policy with DQN
        # With probability epsilon: random action
        # Otherwise: choose action = argmax_a Q(s, a; theta)

        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            # TODO: use the model to compute Q-values and select best action
            action = None  # replace with greedy action from model

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # store transition in replay buffer
        replay_buffer.append((state, action, reward, next_state, done))
        if len(replay_buffer) > buffer_size:
            replay_buffer.pop(0)

        # TODO: if the buffer is large enough, sample a batch and update the network
        if len(replay_buffer) >= batch_size:
            batch = random.sample(replay_buffer, batch_size)
            states_b, actions_b, rewards_b, next_states_b, dones_b = zip(*batch)

            # convert to tensors
            states_tensor = torch.stack([one_hot_state(s) for s in states_b])
            next_states_tensor = torch.stack([one_hot_state(s) for s in next_states_b])
            actions_tensor = torch.tensor(actions_b, dtype=torch.int64).unsqueeze(1)
            rewards_tensor = torch.tensor(rewards_b, dtype=torch.float32).unsqueeze(1)
            dones_tensor = torch.tensor(dones_b, dtype=torch.float32).unsqueeze(1)

            # TODO: compute current Q-values Q(s, a)
            # TODO: compute target Q-values using the Bellman equation
            # TODO: compute loss and update the model

            # Example skeleton (you MUST complete it):
            # q_values = model(states_tensor)
            # q_values = q_values.gather(1, actions_tensor)
            # with torch.no_grad():
            #     next_q_values = model(next_states_tensor).max(1, keepdim=True)[0]
            #     targets = rewards_tensor + gamma * next_q_values * (1 - dones_tensor)
            # loss = criterion(q_values, targets)
            # optimizer.zero_grad()
            # loss.backward()
            # optimizer.step()

        state = next_state
        ep_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    reward_log.append(ep_reward)

    if (ep + 1) % print_every == 0:
        recent_mean = np.mean(reward_log[-print_every:])
        print(f"Episode {ep+1}/{episodes} - epsilon={epsilon:.3f} - recent mean reward={recent_mean:.2f} - buffer size={len(replay_buffer)}")

    # visualization of greedy DQN policy during training
    if (ep + 1) >= dqn_viz_start_ep and (ep + 1) % dqn_viz_every == 0 and len(replay_buffer) >= batch_size:
        print(f"\n[DQN] Visualizing greedy policy at episode {ep+1}")
        v_state, v_info = env.reset()
        for t in range(dqn_viz_max_steps):
            clear_output(wait=True)
            frame = env.render()
            show_frame(frame)
            s_onehot = one_hot_state(v_state)
            with torch.no_grad():
                qvals_v = model(s_onehot)
            v_action = torch.argmax(qvals_v).item()
            v_state, v_reward, v_term, v_trunc, v_info = env.step(v_action)
            if v_term or v_trunc:
                break
            sleep(0.15)

plt.plot(reward_log)
plt.title("DQN: Total reward per episode")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.show()

### 3.4 Visualizing the DQN Policy

Finally, we run one episode using the trained DQN (with **no exploration**) and observe the behavior.

**TODO:**

- Use the trained network to compute Q-values for the current state.
- Always choose the action with the highest Q-value.


In [ ]:
state, info = env.reset()
done = False

for t in range(50):
    clear_output(wait=True)
    frame = env.render()
    show_frame(frame)

    s_onehot = one_hot_state(state)
    with torch.no_grad():
        qvals = model(s_onehot)
    # TODO: choose the greedy action from the DQN output
    action = None  # replace with argmax over qvals

    state, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    sleep(0.15)